#Data Processing

In [1]:
!pip install torch torchvision transformers datasets trl bitsandbytes scikit-learn tqdm modelscope addict simplejson sortedcontainers --break-system-packages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.0/156.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 54.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor
from torchvision import transforms

import sys
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/TechJam



# stream SID trainset because our laptop is too small for it
dataset = load_dataset("saberzl/SID_Set", split="train", streaming=True)

# tested with buffer sizes: [100,500,1000,2500,5000,10000], 10000 seems to be upper bound for what works
shuffled_stream = dataset.shuffle(buffer_size=10000, seed=42)


README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

In [3]:
# ============================================================================
# === CONFIGURATION - ALL SETTINGS IN ONE PLACE ===
# ============================================================================

# --- Model Configuration ---
# image-text-to-text (vision-language) variant -- NOT the text-only "-Base".
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# --- Dataset Configuration ---
#TODO: REPLACE WITH YOUR OWN PATH
DATASET_PATH = ""

# --- Training Configuration (feel free to adjust!) ---
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 4
MAX_STEPS = 500
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.025
LR_SCHEDULER_TYPE = "linear"
OPTIM = "adamw_8bit"  # requires bitsandbytes
SEED = 189

# --- Evaluation Configuration ---
# Answer is a short analysis + \boxed{d}; 128 is plenty and keeps the N_EVAL x2
# greedy generations fast.
EVAL_MAX_NEW_TOKENS = 128  # How many tokens to generate for inference
N_EVAL = 100               # held-out samples reserved from the stream (never trained on)
OUTPUT_DIR = "./aigc_detector_qwen"


### Load base model & tokenizer

In [4]:
# Load the VLM as an image-text-to-text model + its multimodal processor.
# If AutoModelForImageTextToText doesn't resolve for this checkpoint, try
# AutoModelForMultimodalLM (named on the model card) or add trust_remote_code=True.
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype="auto",
)

# Downstream text helpers reference `tokenizer`; the processor bundles one.
tokenizer = processor.tokenizer
# VLM tokenizers may ship without a pad token; SFT batching needs one.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

...

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [5]:
# --- Image preprocessing scaffold (NOT on the text-only training path) -------
# Kept as groundwork for a future vision model. Preprocessing lives in a plain
# .map() over the datasets.IterableDataset from load_dataset(streaming=True) --
# no torch IterableDataset subclass. The stream stays a datasets.IterableDataset,
# which feeds a DataLoader as-is.
_image_transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Ensures matching height and width
    transforms.ToTensor(),          # Converts to tensor scaled 0.0 - 1.0
])


def preprocess_image(example):
    """Per-sample image preprocessing, applied lazily by .map()."""
    image = example["image"]
    if image is not None:
        # Convert 'L' (grayscale) or 'RGBA' to clean 3-channel 'RGB'.
        if image.mode != "RGB":
            image = image.convert("RGB")
        image = _image_transform(image)
    return {"pixel_values": image}


# .map keeps this a datasets.IterableDataset; swap the PIL image for its tensor.
image_stream = shuffled_stream.map(preprocess_image, remove_columns=["image"])


---
### Prompt Construction

Each SID_Set example carries an `image` (PIL) and a `label` (int); the `label`
meaning is defined by the dataset card:

| label | category  | meaning                                        |
| :---: | :-------- | :--------------------------------------------- |
|   0   | Real      | authentic photograph                           |
|   1   | Synthetic | image fully generated by AI                    |
|   2   | Tampered  | real image with AI-manipulated / edited regions |

The model is a vision-language model, so the **real image** is fed alongside the
text as a separate content item (the processor inserts the actual image tokens);
we do **not** splice a `<image>` string into the prompt. The assistant answers
with the **raw integer label** in `\boxed{}` form (e.g. `\boxed{1}`), so no
letter / class-name mapping is needed and 3-class scoring is a direct
`parsed_int == gold_int` comparison.

* **`build_sid_prompt_text`** — the task text (user turn): states the task, lists
  the class options by their integer label, and ends on `Reasoning:` so the model
  continues from there.
* **`build_sid_eval_messages`** — inference messages (image inlined, no assistant
  turn) for a direct `processor.apply_chat_template(...)` call at eval time.
* **`build_sid_train_prompt` / `build_sid_train_completion`** — the training
  sample split into a `prompt` turn (image placeholder + task) and a `completion`
  turn (gold `\boxed{d}` answer). This prompt/completion format lets TRL's vision
  collator mask the prompt and train loss on the answer only
  (`completion_only_loss=True`); real pixels come from the `image` column.


In [6]:
# ============================================================================
# === PROMPT BUILDERS FOR THE STREAMED SID_Set DATASET ===
# (adapted from build_mmlu_prompt / build_mmlu_sft_text in the fine-tuning
#  tutorial; zero-shot, multimodal)
# ============================================================================

# --- Label schema (from the SID_Set dataset card) --------------------------
# Integer label -> (short name, human-readable description). The model answers
# with the raw integer, so there is no option-letter column.
SID_LABELS = {
    0: ("Real",      "an authentic, unmodified photograph"),
    1: ("Synthetic", "an image fully generated by AI (e.g. a diffusion / GAN model)"),
    2: ("Tampered",  "a real image with AI-manipulated or edited regions"),
}


def build_sid_prompt_text() -> str:
    """Task text (user turn) for a single image.

    No `<image>` placeholder string: in the conversational VLM format the image
    is a separate content item and the processor inserts the real image tokens.
    The model answers with the category's integer label in \\boxed{} form,
    parsed back by `parse_label_from_boxed`.
    """
    options = "\n".join(
        f"{label}. {name} \u2014 {desc}"
        for label, (name, desc) in SID_LABELS.items()
    )
    return (
        "You are an expert image-forensics analyst detecting AI-generated images.\n"
        "Classify the image into exactly one category.\n"
        "1. First, give a brief analysis of visual cues such as textures, edges, "
        "lighting, reflections, anatomy, text, and compression artifacts.\n"
        "2. Then, output the final answer as the category number inside a LaTeX box, "
        "e.g., \\boxed{0}.\n\n"
        f"Options:\n{options}\n\n"
        "Reasoning:"  # <--- The model will start generating from here
    )


def build_sid_target(label: int) -> str:
    """Assistant turn (zero-shot): the gold answer as the raw label in \\boxed{}."""
    return f"\\boxed{{{int(label)}}}"


def _sid_user_content(image=None):
    """User content = an image item + the task text.

    `image=None` -> a bare `{"type": "image"}` placeholder (training: real pixels
    come from the dataset's `image` column). `image=<PIL>` -> the pixels inlined
    (eval: a direct `processor.apply_chat_template(...)` call).
    """
    image_item = {"type": "image"} if image is None else {"type": "image", "image": image}
    return [image_item, {"type": "text", "text": build_sid_prompt_text()}]


def build_sid_eval_messages(image):
    """Inference messages for one image (image inlined, no assistant turn)."""
    return [{"role": "user", "content": _sid_user_content(image)}]


def build_sid_train_prompt():
    """Training PROMPT turn (user): image placeholder + task text.

    We use the prompt/completion format (a `prompt` list + a `completion` list),
    NOT a single `messages` list, so TRL's vision collator can mask the prompt and
    compute loss ONLY on the completion (`completion_only_loss=True`). This is the
    answer-only-loss fix for VLMs -- TRL rejects `assistant_only_loss` for vision
    datasets ("Assistant-only loss is not yet supported for vision datasets").
    """
    return [{"role": "user", "content": _sid_user_content(None)}]


def build_sid_train_completion(label: int):
    """Training COMPLETION turn (assistant): the gold \\boxed{d} answer.

    Real pixels come from the dataset's `image` column, matched to the
    `{"type": "image"}` placeholder in the prompt by the VLM SFT collator.
    """
    return [{"role": "assistant", "content": [{"type": "text", "text": build_sid_target(label)}]}]


In [7]:
# --- Sanity check: what the model actually sees -----------------------------
from itertools import islice

print("=== TASK TEXT (user turn) ===")
print(build_sid_prompt_text())

print("\n=== TRAIN PROMPT / COMPLETION (label=1 -> Synthetic) ===")
print("prompt:    ", build_sid_train_prompt())
print("completion:", build_sid_train_completion(1))

# Pull a couple of labels straight off the stream. islice avoids iterating the
# whole 210k-image split.
print("\n=== GOLD TARGETS FOR A FEW STREAMED SAMPLES ===")
for ex in islice(shuffled_stream, 2):
    print(f"label={ex['label']} -> {build_sid_target(ex['label'])}")


=== TASK TEXT (user turn) ===
You are an expert image-forensics analyst detecting AI-generated images.
Classify the image into exactly one category.
1. First, give a brief analysis of visual cues such as textures, edges, lighting, reflections, anatomy, text, and compression artifacts.
2. Then, output the final answer as the category number inside a LaTeX box, e.g., \boxed{0}.

Options:
0. Real — an authentic, unmodified photograph
1. Synthetic — an image fully generated by AI (e.g. a diffusion / GAN model)
2. Tampered — a real image with AI-manipulated or edited regions

Reasoning:

=== TRAIN PROMPT / COMPLETION (label=1 -> Synthetic) ===
prompt:     [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'You are an expert image-forensics analyst detecting AI-generated images.\nClassify the image into exactly one category.\n1. First, give a brief analysis of visual cues such as textures, edges, lighting, reflections, anatomy, text, and compression artifacts.\n2. Then

---
### Evaluation: baseline vs fine-tuned accuracy

To gauge whether fine-tuning helps, we score the model on a **held-out slice of
the stream** (never trained on) both **before** and **after** fine-tuning —
mirroring the tutorial's before/after pattern. Each image is fed through the
processor (real pixels), the model greedy-decodes an answer, and we parse the
`\boxed{d}` label. We report **3-class** accuracy (Real/Synthetic/Tampered) and
the **binary** deliverable metric (AIGC vs authentic).

Order matters: `.train()` mutates `model` in place, so the **baseline eval runs
before** the training cell and its numbers are captured in `baseline`.


#### 3-class vs binary accuracy — what's the difference?

`eval_sid_accuracy` (next cell) reports **two** accuracy numbers for the same
predictions. They differ only in how strict the "correct" test is.

**3-class accuracy** scores the model on the *raw SID_Set label* — did it name
the exact category out of three?

| label | category  |
| :---: | :-------- |
|   0   | Real      |
|   1   | Synthetic (fully AI-generated) |
|   2   | Tampered (real photo, AI-edited regions) |

A prediction is correct only if `pred == gold` exactly (`correct_3class`).
Confusing Synthetic with Tampered is **wrong** here.

**Binary accuracy** first collapses *both* the prediction and the gold label to
the two categories the hackathon actually grades (`collapse_to_binary`):

```
{1 Synthetic, 2 Tampered} -> "AIGC"
{0 Real}                   -> "authentic"
```

then scores `collapse(pred) == collapse(gold)` (`correct_binary`). Now a
Synthetic↔Tampered mix-up is **correct**, because both map to "AIGC".

**Why they differ.** Binary is always **≥** 3-class on the same predictions —
the only errors it forgives are Synthetic↔Tampered (both AIGC). It still
penalizes the mistakes that matter: calling a real photo AI-generated, or the
reverse. Example — gold = 2 (Tampered), model predicts 1 (Synthetic):

- 3-class: **wrong** (1 ≠ 2)
- binary: **right** (both collapse to "AIGC")

**Which one matters.** **Binary is the deliverable metric** — the brief grades
"AIGC vs authentic," so that's the number that reflects the real task. The
**3-class** number is a diagnostic: together with the per-class breakdown it
exposes the common failure mode where a model collapses to one majority class
(that class near 100%, the others near 0%). A large gap (high binary, low
3-class) means the model can tell AI from real but can't separate
fully-synthetic from tampered — which is fine for us, since we collapse them
anyway.


In [8]:
# ============================================================================
# === EVAL HARNESS: parse \boxed{d}, collapse to binary, score accuracy ===
# (adapted from the tutorial's parse_choice_from_boxed / eval_mcq_accuracy +
#  eval_mcq_accuracy_majority)
# ============================================================================
import re
import pandas as pd
from collections import Counter
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix


def parse_label_from_boxed(text):
    """Extract the predicted SID_Set integer label (0/1/2) from generated text.

    Mirrors the tutorial's parse_choice_from_boxed but targets digits: prefer a
    \\boxed{d}; else fall back to the last standalone 0/1/2; else None.
    """
    if text is None:
        return None
    m = re.search(r"\\boxed\{\s*([0-2])\s*\}", text)
    if m:
        return int(m.group(1))
    digits = re.findall(r"\b([0-2])\b", text)
    if digits:
        return int(digits[-1])
    return None


def collapse_to_binary(label):
    """Collapse the 3-class label to the binary deliverable:
    {1 Synthetic, 2 Tampered} -> 'AIGC', {0 Real} -> 'authentic', None -> None."""
    if label is None:
        return None
    return "authentic" if int(label) == 0 else "AIGC"


def _sid_predict(model, processor, image, max_new_tokens, n_votes, temperature, top_p):
    """Predict one image's label (0/1/2 or None) + a sample decoded string.

    n_votes == 1 -> greedy (do_sample=False). n_votes > 1 -> self-consistency:
    sample n_votes answers and take the majority (mode) of the parsed labels,
    dropping unparseable samples (adapted from eval_mcq_accuracy_majority).
    """
    inputs = processor.apply_chat_template(
        build_sid_eval_messages(image),
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    if n_votes == 1:
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        decoded = processor.decode(out[0][prompt_len:], skip_special_tokens=True)
        return parse_label_from_boxed(decoded), decoded

    # Majority voting: one batched call with num_return_sequences. If the VLM
    # can't expand image features that way, fall back to n_votes separate calls.
    try:
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, top_p=top_p, num_return_sequences=n_votes,
        )
        seqs = [out[i][prompt_len:] for i in range(out.shape[0])]
    except Exception:
        seqs = [
            model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                           temperature=temperature, top_p=top_p)[0][prompt_len:]
            for _ in range(n_votes)
        ]

    votes, last_decoded = [], ""
    for seq in seqs:
        last_decoded = processor.decode(seq, skip_special_tokens=True)
        v = parse_label_from_boxed(last_decoded)
        if v is not None:            # ignore unparseable samples
            votes.append(v)
    pred = Counter(votes).most_common(1)[0][0] if votes else None
    return pred, last_decoded


def print_confusion(cm, cm_labels, title=""):
    """Pretty-print a small confusion matrix (rows=gold, cols=pred)."""
    names = {0: "Real", 1: "Synth", 2: "Tamp", -1: "None"}
    print((title + "  (rows=gold, cols=pred)").strip())
    print("        " + "  ".join(f"{names[c]:>5s}" for c in cm_labels))
    for i, gl in enumerate(cm_labels):
        if gl == -1:                 # gold is never the -1 "unparsed" bucket
            continue
        row = "  ".join(f"{cm[i][j]:5d}" for j in range(len(cm_labels)))
        print(f"  {names[gl]:>5s} {row}")


@torch.no_grad()
def eval_sid_accuracy(model, processor, eval_samples,
                      max_new_tokens=EVAL_MAX_NEW_TOKENS,
                      n_votes=1, temperature=0.7, top_p=0.9):
    """Score held-out images: 3-class + binary acc, macro-F1, balanced acc, confusion.

    Adapted from the tutorial's eval_mcq_accuracy: model.eval() + no_grad, one
    image per call, answer parsed from \\boxed{}. Feeds the REAL image through the
    processor. `n_votes > 1` switches to majority voting (self-consistency).
    Macro-F1 / balanced accuracy are collapse-aware: a constant predictor scores
    near zero on them even when its 3-class accuracy looks respectable.
    """
    model.eval()
    records = []
    for idx, ex in enumerate(eval_samples):
        gold = int(ex["label"])
        pred, decoded = _sid_predict(
            model, processor, ex["image"], max_new_tokens, n_votes, temperature, top_p
        )
        records.append({
            "idx": idx,
            "gold": gold,
            "parsed": pred,
            "decoded": decoded,
            "correct_3class": pred is not None and pred == gold,
            "correct_binary": collapse_to_binary(pred) == collapse_to_binary(gold),
        })
        if (idx + 1) % 10 == 0:
            print(f"Evaluated {idx + 1}/{len(eval_samples)}...")

    details = pd.DataFrame(records)
    acc_3class = float(details["correct_3class"].mean())
    acc_binary = float(details["correct_binary"].mean())
    # Per-class 3-class accuracy so a collapse-to-majority failure is visible.
    per_class = details.groupby("gold")["correct_3class"].agg(["mean", "count"]).to_dict("index")

    # Collapse-aware metrics. Unparsed preds -> sentinel -1 so they count as wrong
    # (and appear as an extra "None" column in the confusion matrix).
    gold_arr = details["gold"].tolist()
    pred_arr = [p if p is not None else -1 for p in details["parsed"]]
    macro_f1 = float(f1_score(gold_arr, pred_arr, labels=[0, 1, 2], average="macro", zero_division=0))
    balanced_acc = float(balanced_accuracy_score(gold_arr, pred_arr))
    cm_labels = [0, 1, 2, -1]
    confusion = confusion_matrix(gold_arr, pred_arr, labels=cm_labels)

    tag = "greedy" if n_votes == 1 else f"majority@{n_votes}"
    print(f"[{tag}] 3-class {acc_3class * 100:.2f}%  binary {acc_binary * 100:.2f}%  "
          f"macroF1 {macro_f1:.3f}  balAcc {balanced_acc * 100:.2f}%  (n={len(details)})")
    return {
        "acc_3class": acc_3class,
        "acc_binary": acc_binary,
        "macro_f1": macro_f1,
        "balanced_acc": balanced_acc,
        "per_class": per_class,
        "confusion": confusion,
        "cm_labels": cm_labels,
        "details": details,
    }


In [9]:
# ============================================================================
# === BUILD HELD-OUT EVAL SET + BASELINE (pre-fine-tune) ACCURACY ===
# ============================================================================
# Reserve a disjoint eval slice: the first N_EVAL of the shuffled stream. The
# training cell consumes shuffled_stream.skip(N_EVAL), so eval is never trained
# on. Materialise once so baseline and fine-tuned score the SAME samples.
eval_samples = list(shuffled_stream.take(N_EVAL))
print(f"Held-out eval samples: {len(eval_samples)}")

# BASELINE: evaluate the pre-fine-tuned model. MUST run before sid_trainer.train()
# (below), which mutates `model` in place.
baseline = eval_sid_accuracy(model, processor, eval_samples, EVAL_MAX_NEW_TOKENS)


Held-out eval samples: 100
Evaluated 10/100...
Evaluated 20/100...
Evaluated 30/100...
Evaluated 40/100...
Evaluated 50/100...
Evaluated 60/100...
Evaluated 70/100...
Evaluated 80/100...
Evaluated 90/100...
Evaluated 100/100...
[greedy] 3-class 21.00%  binary 70.00%  macroF1 0.166  balAcc 20.23%  (n=100)


---
### Map prompt/completion into the stream + fine-tune with `SFTTrainer`

Following the tutorial's pattern (`SFTConfig` / `SFTTrainer`) rather than a
hand-rolled loop, but on the **multimodal** path:

1. **Reserve a held-out eval slice first.** `shuffled_stream.take(N_EVAL)` is
   evaluated (baseline) before training; training consumes
   `shuffled_stream.skip(N_EVAL)`, so the two never overlap — we never train on
   the eval samples.
2. **Map prompt/completion into the stream** — each streamed example gets a
   `"prompt"` (user turn, image placeholder + task) and a `"completion"`
   (assistant `\boxed{d}` answer), keeping the `image` column. TRL detects the
   `image` column and uses its native `DataCollatorForVisionLanguageModeling`,
   processing pixels on the fly.
3. Configure `SFTConfig` with `max_length=None` (so image tokens aren't
   truncated), `max_steps=MAX_STEPS` (the source is a streaming
   `IterableDataset` with no length), and **`completion_only_loss=True`** so loss
   is computed on the answer only (the collator masks the prompt to `-100`).
   Hand the stream to `SFTTrainer` with `processing_class=processor` and call
   `.train()`.

> **Notes.** The image is genuinely fed to the model (vision pathway), and loss
> falls only on the `\boxed{d}` answer — this is the fix for the earlier
> collapse-to-one-class failure, where full-sequence loss over an identical
> prompt swamped the answer signal. TRL does not support `assistant_only_loss`
> for vision datasets, hence the prompt/completion + `completion_only_loss` route.
> `OPTIM="adamw_8bit"` requires `bitsandbytes` (installed above).


In [10]:
# ============================================================================
# === MAP PROMPT/COMPLETION INTO THE STREAM (multimodal VLM SFT) ===
# ============================================================================
# Train on everything AFTER the reserved eval slice, so training never sees the
# held-out samples that the baseline/fine-tuned eval scores (defined above).
train_source = shuffled_stream.skip(N_EVAL)

# .map adds "prompt" + "completion" fields lazily and drops the raw label. The
# `image` column is kept: TRL detects it (vision dataset) and uses
# DataCollatorForVisionLanguageModeling with completion_only_loss, masking the
# prompt so loss falls only on the \boxed{d} answer.
sft_stream = train_source.map(
    lambda ex: {
        "prompt": build_sid_train_prompt(),
        "completion": build_sid_train_completion(ex["label"]),
    },
    remove_columns=["label"],
)

# Peek: confirm each sample now carries image + prompt + completion.
_peek = next(iter(sft_stream))
print("keys:", list(_peek.keys()))
print("prompt:    ", _peek["prompt"])
print("completion:", _peek["completion"])


keys: ['img_id', 'image', 'mask', 'width', 'height', 'prompt', 'completion']
prompt:     [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'You are an expert image-forensics analyst detecting AI-generated images.\nClassify the image into exactly one category.\n1. First, give a brief analysis of visual cues such as textures, edges, lighting, reflections, anatomy, text, and compression artifacts.\n2. Then, output the final answer as the category number inside a LaTeX box, e.g., \\boxed{0}.\n\nOptions:\n0. Real — an authentic, unmodified photograph\n1. Synthetic — an image fully generated by AI (e.g. a diffusion / GAN model)\n2. Tampered — a real image with AI-manipulated or edited regions\n\nReasoning:'}]}]
completion: [{'role': 'assistant', 'content': [{'type': 'text', 'text': '\\boxed{0}'}]}]


In [11]:
# ============================================================================
# === SET UP SFTTrainer + FINE-TUNE (adapted from the fine-tuning tutorial) ===
# ============================================================================
# NB: run the baseline-eval cell above BEFORE this cell -- .train() mutates
# `model` in place, so the pre-fine-tuned numbers must be captured first.
sid_sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    # Streaming IterableDataset has no length -> drive with max_steps, not epochs.
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    optim=OPTIM,               # "adamw_8bit" needs bitsandbytes; else "adamw_torch"
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    seed=SEED,
    report_to="none",
    # VLM specifics: never truncate (would drop image tokens); keep the image
    # column so the vision collator receives it.
    max_length=None,
    remove_unused_columns=False,
    # Fix for the collapse-to-one-class failure: supervise ONLY the \boxed{d}
    # answer, not the identical prompt shared by every example. Without this the
    # full-sequence loss is dominated by the repeated prompt and the one-digit
    # answer signal is swamped -> the model degenerates to a constant class.
    # TRL rejects assistant_only_loss for vision datasets, so we use the
    # prompt/completion format (mapped above) + completion_only_loss: the vision
    # collator sets the prompt-part labels to -100.
    completion_only_loss=True,
)

sid_trainer = SFTTrainer(
    model=model,
    args=sid_sft_config,
    train_dataset=sft_stream,
    eval_dataset=None,
    processing_class=processor,
)

sid_trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss
1,1.340552
2,1.560916
3,0.933110
4,0.282192
5,0.154002
6,0.166310
7,0.118079
8,0.177524
9,0.056590
10,0.102033


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.06975227803008056, metrics={'train_runtime': 5124.7133, 'train_samples_per_second': 0.39, 'train_steps_per_second': 0.098, 'total_flos': 7741615638262272.0, 'train_loss': 0.06975227803008056, 'epoch': 1.0})

---
### Results: did fine-tuning help?

Re-run the eval on the fine-tuned `model` and print the side-by-side comparison.
Alongside 3-class / binary accuracy we report **collapse-aware** metrics:

- **macro-F1** and **balanced accuracy** — averaged over the three classes, so a
  model that collapses to one class scores near the floor (~0.15 macro-F1) no
  matter how good its raw 3-class % looks. These are the honest model-selection
  numbers.
- a **3×3 confusion matrix** (rows = gold, cols = pred, plus a `None`/unparsed
  column) — a single populated column is the visual signature of collapse.

We also run **majority voting** (`mv@5`: sample 5 answers, take the mode) on the
fine-tuned model. It reduces variance for a noisy-but-informative model, but
**cannot** rescue a fully collapsed one — so it is reported next to greedy, not
instead of it. A positive Δ binary / Δ macro-F1 means fine-tuning genuinely
helped.


In [12]:
# ============================================================================
# === FINE-TUNED ACCURACY + BASELINE-VS-FINE-TUNED COMPARISON ===
# ============================================================================
# Same held-out samples, greedy eval on the now-fine-tuned `model`...
finetuned = eval_sid_accuracy(model, processor, eval_samples, EVAL_MAX_NEW_TOKENS)
# ...plus self-consistency: majority vote over several sampled answers. This cuts
# variance for a noisy-but-informative model; it CANNOT rescue a fully collapsed
# one (every vote would be the same class), so read it alongside greedy.
finetuned_mv = eval_sid_accuracy(
    model, processor, eval_samples, EVAL_MAX_NEW_TOKENS,
    n_votes=5, temperature=0.7, top_p=0.9,
)


def _row(name, r):
    return (f"{name:18s} | 3cls {r['acc_3class'] * 100:5.2f}%  bin {r['acc_binary'] * 100:5.2f}%  "
            f"macroF1 {r['macro_f1']:.3f}  balAcc {r['balanced_acc'] * 100:5.2f}%")


print("\n=== Baseline vs fine-tuned (macro-F1/balanced-acc are collapse-aware) ===")
print(_row("baseline", baseline))
print(_row("finetuned (greedy)", finetuned))
print(_row("finetuned (mv@5)", finetuned_mv))
print(f"\nΔ binary  (greedy): {(finetuned['acc_binary'] - baseline['acc_binary']) * 100:+.2f} pts")
print(f"Δ macroF1 (greedy): {finetuned['macro_f1'] - baseline['macro_f1']:+.3f}   "
      f"(a constant single-class predictor floors macro-F1 at ~0.15)")

print("\nPer-class 3-class accuracy (gold -> baseline -> finetuned greedy):")
for lbl in sorted(SID_LABELS):
    name = SID_LABELS[lbl][0]
    b = baseline["per_class"].get(lbl, {"mean": float("nan"), "count": 0})
    f = finetuned["per_class"].get(lbl, {"mean": float("nan"), "count": 0})
    print(f"  {lbl} {name:9s} | {b['mean'] * 100:5.1f}%  ->  {f['mean'] * 100:5.1f}%  (n={b['count']})")

print()
print_confusion(baseline["confusion"], baseline["cm_labels"], "Baseline confusion")
print()
print_confusion(finetuned["confusion"], finetuned["cm_labels"], "Fine-tuned (greedy) confusion")


Evaluated 10/100...
Evaluated 20/100...
Evaluated 30/100...
Evaluated 40/100...
Evaluated 50/100...
Evaluated 60/100...
Evaluated 70/100...
Evaluated 80/100...
Evaluated 90/100...
Evaluated 100/100...
[greedy] 3-class 96.00%  binary 98.00%  macroF1 0.959  balAcc 95.76%  (n=100)
Evaluated 10/100...
Evaluated 20/100...
Evaluated 30/100...
Evaluated 40/100...
Evaluated 50/100...
Evaluated 60/100...
Evaluated 70/100...
Evaluated 80/100...
Evaluated 90/100...
Evaluated 100/100...
[majority@5] 3-class 96.00%  binary 98.00%  macroF1 0.959  balAcc 95.76%  (n=100)

=== Baseline vs fine-tuned (macro-F1/balanced-acc are collapse-aware) ===
baseline           | 3cls 21.00%  bin 70.00%  macroF1 0.166  balAcc 20.23%
finetuned (greedy) | 3cls 96.00%  bin 98.00%  macroF1 0.959  balAcc 95.76%
finetuned (mv@5)   | 3cls 96.00%  bin 98.00%  macroF1 0.959  balAcc 95.76%

Δ binary  (greedy): +28.00 pts
Δ macroF1 (greedy): +0.793   (a constant single-class predictor floors macro-F1 at ~0.15)

Per-class 3-cla

---
### WildFake validation benchmark (reference-only)

The brief supplies a **demo subset of WildFake** to track progress — it is *not*
scored and **must never be trained on**:

| bucket | source | count |
| :--- | :--- | ---: |
| Non-AIGC (authentic) | COCO val2017 | 4998 |
| AIGC | DALL·E Advanced | 8843 |

We pull it from ModelScope (`hy2628982280/WildFake`) **streamed**
(`use_streaming=True`) — same disk-light approach as the SID_Set training
stream. The repo hosts the *full* WildFake, so the cell below **filters** to the
two demo buckets using the label columns confirmed from the dataset's CSVs:

- **`IsFake`** — the real/fake flag (`0` real, `1` AI-generated).
- **`Architecture` / `Image_path`** — source; `real_coco` rows carry `coco`,
  DALL·E rows carry `DALLE`.
- **`IsAdvanced`** — marks the "Advanced" split, so DALL·E Advanced =
  `IsFake=1 & DALLE & IsAdvanced=1`.

Metric is **binary only** (AIGC vs authentic) — the deliverable target. Each
scored image yields a `pred` AIGC likelihood (`1.0`/`0.0`) plus its
`image_path`, mirroring the final JSON output format. The model's 3-class
`\boxed{d}` prediction is collapsed to binary via `collapse_to_binary` from the
eval harness above.

> The cell first **peeks one streamed row** and prints its keys. Confirm the row
> exposes the expected columns and how it carries pixels (a decoded image vs a
> bare `Image_path`); if it's path-only, set `WF_IMAGES_ROOT` to the downloaded
> Images root so paths resolve. Then uncomment the final line to run.


In [21]:
# ============================================================================
# === WILDFAKE VALIDATION BENCHMARK (reference-only, streamed, BINARY) ===
# COCO val2017 non-AIGC + DALL·E Advanced AIGC -- the brief's demo subset.
# NEVER trained on: this is a held-out reference benchmark (problem brief).
# Reuses the model's prediction path (_sid_predict) + collapse_to_binary above.
# ============================================================================
from modelscope.msdatasets import MsDataset
from PIL import Image
import os

# --- Schema confirmed from the dataset's label CSVs -------------------------
# Columns: Generator, Architecture, Weight, Category, IsAdvanced, IsFake,
#          Image_path, Num.  Example DALL-E-3 row:
#   Generator=Diffusion_based  Architecture=DALLE  IsAdvanced=1  IsFake=1
#   Image_path=./Diffusion_based/DALLE/Advanced/DALLE3/.../xxx.jpg
# => IsFake is the real/fake flag; Architecture=="DALLE" & IsAdvanced==1 is the
#    "DALL-E Advanced" AIGC bucket; COCO reals sit in real_coco (path has "coco").
# Values may arrive as ints or strings ("1"/"0"/"True") -> coerce defensively.

WF_DATASET = "hy2628982280/WildFake"
WF_SUBSET  = "default"
WF_SPLIT   = "train"
# Cap per class for a quick pass; raise toward the full 4998 / 8843 for the
# official demo numbers (None = take every matching row).
WF_MAX_AUTHENTIC = 200
WF_MAX_AIGC      = 200
# Streamed rows may carry only Image_path (a relative "./..."), not decoded
# pixels. If so, point this at the local Images root so paths resolve; the peek
# below reveals which case we're in. None = assume the row already has an image.
WF_IMAGES_ROOT = None


def _wf_truthy(v):
    """Coerce IsFake / IsAdvanced ('1'/'0'/1/0/'True'/'False') to bool."""
    if isinstance(v, str):
        return v.strip().lower() in {"1", "true", "yes", "y", "t"}
    return bool(v)


def _wf_haystack(ex):
    """Lower-cased blob of the source-identifying fields, for substring tests."""
    return " ".join(
        str(ex.get(k, "")) for k in ("Architecture", "Category", "Generator", "Image_path")
    ).lower()


def _wf_is_authentic_coco(ex):
    """Non-AIGC COCO real image: not fake AND source mentions coco."""
    return (not _wf_truthy(ex.get("IsFake"))) and "coco" in _wf_haystack(ex)


def _wf_is_dalle_advanced(ex):
    """AIGC DALL-E Advanced: fake AND DALLE source AND IsAdvanced."""
    return (
        _wf_truthy(ex.get("IsFake"))
        and "dalle" in _wf_haystack(ex)
        and _wf_truthy(ex.get("IsAdvanced"))
    )


def _wf_image(ex):
    """Return a PIL.Image (RGB) for a streamed row, or None if unresolvable.

    Prefers a decoded image the loader already provides; else opens Image_path
    (joined under WF_IMAGES_ROOT when the path is relative).
    """
    for k in ("image", "Image", "img"):
        obj = ex.get(k)
        if isinstance(obj, Image.Image):
            return obj.convert("RGB")
    path = ex.get("Image_path")
    if path:
        if WF_IMAGES_ROOT:
            path = os.path.join(WF_IMAGES_ROOT, str(path).lstrip("./"))
        try:
            return Image.open(path).convert("RGB")
        except Exception:
            return None
    return None


# --- Peek one streamed row so the ACTUAL keys are visible on Colab -----------
# (schema above is from the label CSVs; confirm the streamed row exposes the
#  same keys -- and whether it carries decoded pixels or just Image_path.)
_wf_peek_stream = MsDataset.load(
    WF_DATASET, subset_name=WF_SUBSET, split=WF_SPLIT, use_streaming=True,
)
_wf_peek = next(iter(_wf_peek_stream))
print("WildFake row keys:", list(_wf_peek.keys()))
print("sample row:", {k: _wf_peek[k] for k in list(_wf_peek)[:8]})


@torch.no_grad()
def eval_wildfake_binary(model, processor,
                         max_authentic=WF_MAX_AUTHENTIC, max_aigc=WF_MAX_AIGC,
                         max_new_tokens=EVAL_MAX_NEW_TOKENS):
    """Binary AIGC-vs-authentic eval over the streamed WildFake demo subset.

    Streams the split fresh, keeps up to `max_authentic` COCO reals + `max_aigc`
    DALL-E-Advanced fakes, greedy-predicts each (3-class \\boxed{d}), collapses
    to binary, and reports accuracy + a per-class breakdown. Each record's `pred`
    is the model's AIGC likelihood (1.0 = AIGC, 0.0 = authentic) -- the field the
    final deliverable JSON needs, alongside `image_path`.
    """
    model.eval()
    stream = MsDataset.load(
        WF_DATASET, subset_name=WF_SUBSET, split=WF_SPLIT, use_streaming=True,
    )
    want = {"authentic": max_authentic, "AIGC": max_aigc}
    got  = {"authentic": 0, "AIGC": 0}

    def _full(c):
        return want[c] is not None and got[c] >= want[c]

    records = []
    for ex in stream:
        if _wf_is_authentic_coco(ex):
            gold = "authentic"
        elif _wf_is_dalle_advanced(ex):
            gold = "AIGC"
        else:
            continue
        if _full(gold):
            if _full("authentic") and _full("AIGC"):
                break
            continue                       # this class done; keep filling the other
        img = _wf_image(ex)
        if img is None:                    # pixels unresolved -> skip (see WF_IMAGES_ROOT)
            continue
        label3, _ = _sid_predict(model, processor, img, max_new_tokens, 1, 0.7, 0.9)
        pred_bin = collapse_to_binary(label3)
        got[gold] += 1
        records.append({
            "image_path": ex.get("Image_path"),
            "gold": gold,
            "pred": 1.0 if pred_bin == "AIGC" else 0.0,   # AIGC likelihood (deliverable field)
            "pred_bin": pred_bin,
            "correct": pred_bin == gold,
        })
        if len(records) % 20 == 0:
            print(f"scored {len(records)}  (authentic {got['authentic']}, AIGC {got['AIGC']})")

    details = pd.DataFrame(records)
    if len(details) == 0:
        print("No samples scored -- check the peek above (keys / how pixels are "
              "exposed) and set WF_IMAGES_ROOT if rows carry only Image_path.")
        return {"acc_binary": float("nan"), "per_class": {}, "details": details}

    acc = float(details["correct"].mean())
    per_class = details.groupby("gold")["correct"].agg(["mean", "count"]).to_dict("index")
    print(f"\nWildFake binary accuracy: {acc * 100:.2f}%  (n={len(details)})")
    for cls in ("authentic", "AIGC"):
        c = per_class.get(cls, {"mean": float("nan"), "count": 0})
        print(f"  {cls:9s}: {c['mean'] * 100:5.1f}%  (n={c['count']})")
    return {"acc_binary": acc, "per_class": per_class, "details": details}


# Uncomment to run once the peek confirms keys / pixel access:
# wf_result = eval_wildfake_binary(model, processor)


2026-08-27 15:25:41,028 - modelscope - INFO - Some files matched the pattern 'hf://datasets/hy2628982280/WildFake@master/**/*[-._ 0-9]train[-._ 0-9]*/**' but don't have valid data file extensions: ['hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_time_other.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_gan.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_generator.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/cross_time_split_gan.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_other.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_time_sd.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/train_test_split.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/cross_generator_split.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_midjourney.py', 'h

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

2026-08-27 15:25:41,061 - modelscope - INFO - Some files matched the pattern 'hf://datasets/hy2628982280/WildFake@master/**/*[-._ 0-9]test/**' but don't have valid data file extensions: ['hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_time_other.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_gan.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_generator.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/cross_time_split_gan.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_other.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_time_sd.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/train_test_split.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/cross_generator_split.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_midjourney.py', 'hf://dataset

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

WildFake row keys: ['Generator', 'Architecture', 'Weight', 'Category', 'IsAdvanced', 'IsFake', 'Image_path', 'Num']
sample row: {'Generator': 'Diffusion_based', 'Architecture': 'ADM', 'Weight': 'ADM', 'Category': 'ADM', 'IsAdvanced': 0, 'IsFake': 1, 'Image_path': './Diffusion_based/ADM/imgs/0000af666a76220ad33abaa4ce80dd22.png', 'Num': 1}


2026-08-27 15:26:02,460 - modelscope - INFO - Some files matched the pattern 'hf://datasets/hy2628982280/WildFake@master/**/*[-._ 0-9]train[-._ 0-9]*/**' but don't have valid data file extensions: ['hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_time_other.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_gan.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_generator.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/cross_time_split_gan.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_other.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_time_sd.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/train_test_split.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/cross_generator_split.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_midjourney.py', 'h

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

2026-08-27 15:26:02,493 - modelscope - INFO - Some files matched the pattern 'hf://datasets/hy2628982280/WildFake@master/**/*[-._ 0-9]test/**' but don't have valid data file extensions: ['hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_time_other.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_gan.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_generator.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/cross_time_split_gan.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_other.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/add_real_cross_time_sd.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/train_test_split.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/cross_generator_split.py', 'hf://datasets/hy2628982280/WildFake@master/split_train_test/count_cross_time_midjourney.py', 'hf://dataset

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
### Local WildFake test-set eval (preferred over streaming)

WildFake stores its pixels **inside multi-GB per-category zips**, keyed by
`Image_path` — so `MsDataset.load(streaming=True)` (the cell above) tends to
hand back path strings, not decoded images. The reliable path is to pull a small
test set to disk with `pull_wildfake_balanced.py` (HTTP range requests, no full
download) and score that folder directly.

This cell reads `wildfake_balanced/labels.csv`
(`image_path, category, generator, label`, with **label 0 = real / 1 = fake**),
loads each image, runs the same greedy `_sid_predict` → `collapse_to_binary`
prediction path used everywhere else, and reports:

- **overall binary accuracy** (AIGC vs authentic — the deliverable metric),
- **per-class** accuracy (authentic vs AIGC — a collapse flattens one to ~0),
- **per-category** accuracy, worst-first (which *generators* fool the model;
  the set is category-balanced so these compare directly).

It also writes **`wildfake_balanced_preds.json`** in the deliverable shape
(`[{image_path, pred}]`, `pred` = AIGC likelihood). Set `TEST_DIR` if your folder
name differs; the Colab bootstrap cell `%cd`s into `TechJam`, so the default
resolves there. Reference-only — **never trained on**.


In [22]:
# ============================================================================
# === LOCAL WILDFAKE TEST-SET EVAL (labeled folder + labels.csv, BINARY) ===
# Scores a locally-pulled WildFake test set (from pull_wildfake_balanced.py): a
# folder of category subdirs + labels.csv (image_path relative to the folder,
# label 0=real / 1=fake). Disk-light + reproducible -- preferred over MsDataset
# streaming, since WildFake stores pixels inside multi-GB zips, not decoded
# stream rows. Reuses _sid_predict + collapse_to_binary from the eval harness.
# NEVER trained on: reference-only test set (problem brief).
# ============================================================================
import os
import json
from PIL import Image

# TEST_DIR is relative to the working dir. The Colab bootstrap cell %cd's into
# TechJam, so "wildfake_balanced" resolves there (and locally too).
TEST_DIR   = "wildfake_balanced"
LABELS_CSV = os.path.join(TEST_DIR, "labels.csv")
OUT_JSON   = "wildfake_balanced_preds.json"   # deliverable-shaped: [{image_path, pred}]


def _bin_from_label(label) -> str:
    """WildFake test label -> binary: 0 -> authentic, anything else -> AIGC.
    Matches collapse_to_binary's real/AIGC split so gold and pred are comparable."""
    return "authentic" if int(label) == 0 else "AIGC"


@torch.no_grad()
def eval_local_folder(model, processor, test_dir=TEST_DIR, labels_csv=LABELS_CSV,
                      max_new_tokens=EVAL_MAX_NEW_TOKENS, out_json=OUT_JSON):
    """Binary AIGC-vs-authentic eval over a local labeled image folder.

    Reads labels_csv (columns include `image_path` relative to test_dir and a
    `label` 0=real / 1=fake), greedy-predicts each image (3-class \\boxed{d}),
    collapses to binary, and reports overall / per-class / per-category accuracy.
    Writes deliverable-shaped predictions ([{image_path, pred}]) to out_json.
    """
    model.eval()
    df = pd.read_csv(labels_csv)
    print(f"Loaded {len(df)} labeled rows from {labels_csv}")

    records = []
    for i, row in enumerate(df.itertuples(index=False)):
        gold = _bin_from_label(row.label)
        cat = getattr(row, "category", "")
        path = os.path.join(test_dir, row.image_path)
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"skip (can't open) {path}: {e}")
            continue
        label3, _ = _sid_predict(model, processor, img, max_new_tokens, 1, 0.7, 0.9)
        pred_bin = collapse_to_binary(label3)
        records.append({
            "image_path": path,
            "category": cat,
            "gold": gold,
            "pred": 1.0 if pred_bin == "AIGC" else 0.0,   # AIGC likelihood (deliverable field)
            "pred_bin": pred_bin,
            "parsed": label3,
            "correct": pred_bin == gold,
        })
        if (i + 1) % 20 == 0:
            print(f"scored {i + 1}/{len(df)}...")

    details = pd.DataFrame(records)
    acc = float(details["correct"].mean())
    n_unparsed = int(details["parsed"].isna().sum())

    # Deliverable JSON: only image_path + pred, one row per scored image.
    with open(out_json, "w") as f:
        json.dump(
            [{"image_path": r["image_path"], "pred": r["pred"]} for r in records],
            f, indent=2,
        )

    print(f"\nLocal test-set binary accuracy: {acc * 100:.2f}%  "
          f"(n={len(details)}, unparsed={n_unparsed})")
    # Per-class (authentic vs AIGC): a collapse would flatten one of these to ~0.
    for cls in ("authentic", "AIGC"):
        sub = details[details["gold"] == cls]
        if len(sub):
            print(f"  {cls:9s}: {sub['correct'].mean() * 100:5.1f}%  (n={len(sub)})")
    # Per-category (which generators fool the model). The set is category-
    # balanced, so these are directly comparable; sorted worst-first.
    if "category" in details and details["category"].astype(bool).any():
        print("\nPer-category accuracy (worst-first):")
        by_cat = details.groupby("category")["correct"].agg(["mean", "count"])
        for cat, r in by_cat.sort_values("mean").iterrows():
            print(f"  {cat:<16} {r['mean'] * 100:5.1f}%  (n={int(r['count'])})")
    print(f"\nWrote deliverable predictions -> {out_json}")
    return {"acc_binary": acc, "details": details}


# Uncomment to run:
# local_result = eval_local_folder(model, processor)


Loaded 260 labeled rows from wildfake_balanced/labels.csv
scored 20/260...
scored 40/260...
scored 60/260...
scored 80/260...
scored 100/260...
scored 120/260...
scored 140/260...
scored 160/260...
scored 180/260...
scored 200/260...
scored 220/260...
scored 240/260...
scored 260/260...

Local test-set binary accuracy: 65.77%  (n=260, unparsed=0)
  authentic:  88.3%  (n=120)
  AIGC     :  46.4%  (n=140)

Per-category accuracy (worst-first):
  VQDM               0.0%  (n=20)
  ADM               35.0%  (n=20)
  DDPM              40.0%  (n=20)
  Imagen            45.0%  (n=20)
  DDIM              60.0%  (n=20)
  SDwithAdaptor     65.0%  (n=20)
  laion5b           65.0%  (n=20)
  imagenet          65.0%  (n=20)
  Midjourney        80.0%  (n=20)
  afhq             100.0%  (n=20)
  church           100.0%  (n=20)
  celebahq         100.0%  (n=20)
  ffhq             100.0%  (n=20)

Wrote deliverable predictions -> wildfake_balanced_preds.json


In [ ]:
# ============================================================================
# === WILDFAKE LOCAL TEST-SET ACCURACY + CROSS-DATASET COMPARISON ===
# ============================================================================
# Score the local WildFake test set (greedy). eval_local_folder prints overall /
# per-class / per-category accuracy and writes wildfake_balanced_preds.json.
wildfake = eval_local_folder(model, processor)

# Context: WildFake is a CROSS-DATASET test -- different generators + reals than
# SID_Set -- so compare its binary accuracy against the in-distribution SID_Set
# fine-tuned binary from the comparison cell above. A large drop = source shift.
print("\n=== In-distribution (SID_Set) vs cross-dataset (WildFake) -- binary ===")
if "finetuned" in globals():
    sid_bin = finetuned["acc_binary"]
    wf_bin = wildfake["acc_binary"]
    print(f"  SID_Set held-out (fine-tuned, greedy) : {sid_bin * 100:5.2f}%")
    print(f"  WildFake local test set (greedy)      : {wf_bin * 100:5.2f}%")
    print(f"  Δ (WildFake - SID_Set)                : {(wf_bin - sid_bin) * 100:+.2f} pts")
else:
    print("  (run the fine-tuned comparison cell above first for the SID_Set number)")


---
### Save / reload the fine-tuned model

Colab's local disk is wiped when the runtime ends, so persist the fine-tuned
weights to the **Drive-mounted `TechJam` folder** before closing. `save_pretrained`
writes a self-contained folder (`config.json` + safetensors weights + processor /
tokenizer) that `from_pretrained` can reload — for submission or for scoring other
test sets.

- **Save cell** — writes `aigc_detector_qwen/` to Drive (survives the runtime
  closing). Uncomment the `shutil.make_archive` line to also get a single
  `.zip` for submission/download.
- **Reload cell** — rebuilds `model` / `processor` / `tokenizer` in a **fresh**
  session (run the pip-install + Drive-mount bootstrap first). Every eval
  function above then works unchanged.

> This saves the merged full weights (a plain `save_pretrained`), not just a
> LoRA adapter — we fine-tuned all params, so the folder is a complete model.


In [23]:
# ============================================================================
# === SAVE FINE-TUNED MODEL + PROCESSOR (persist before the runtime closes) ===
# ============================================================================
# Colab wipes local disk when the runtime ends. Save to the Drive-mounted TechJam
# folder (cwd after the bootstrap %cd) so the weights survive and can be reloaded
# or submitted. save_pretrained writes config + weights (safetensors) + the
# processor / tokenizer files -- everything from_pretrained needs to reload.
import os

SAVE_DIR = "aigc_detector_qwen"   # relative to TechJam on Drive -> persists

model.save_pretrained(SAVE_DIR, safe_serialization=True)   # config.json + *.safetensors
processor.save_pretrained(SAVE_DIR)                         # processor + tokenizer files

files = sorted(os.listdir(SAVE_DIR))
size_mb = sum(os.path.getsize(os.path.join(SAVE_DIR, f))
              for f in files if os.path.isfile(os.path.join(SAVE_DIR, f))) / 1e6
print(f"Saved fine-tuned model -> {os.path.abspath(SAVE_DIR)}  ({size_mb:.1f} MB)")
print("files:", files)

# Optional: bundle into one zip for submission / download.
# import shutil; shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)
# print("zipped ->", SAVE_DIR + ".zip")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model -> /content/drive/.shortcut-targets-by-id/1R_6rD27OAywNJf8bhcHp_1NC-Pf8rfzc/TechJam/aigc_detector_qwen  (1726.0 MB)
files: ['chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json']


In [24]:
# ============================================================================
# === RELOAD THE SAVED MODEL (fresh runtime / other test sets) ===
# ============================================================================
# Run this in a NEW session AFTER the pip-install + Drive-mount bootstrap cells.
# It rebuilds `model`, `processor`, `tokenizer` from the saved folder so the eval
# functions (eval_local_folder / eval_sid_accuracy / eval_wildfake_binary) work
# unchanged. NB: in the CURRENT session the model is already loaded -- running
# this reloads a SECOND copy onto the GPU (extra memory), so it's mainly meant
# for a fresh runtime.
from transformers import AutoModelForImageTextToText, AutoProcessor

SAVE_DIR = "aigc_detector_qwen"
processor = AutoProcessor.from_pretrained(SAVE_DIR)
model = AutoModelForImageTextToText.from_pretrained(
    SAVE_DIR, device_map="auto", dtype="auto",
)
tokenizer = processor.tokenizer
if tokenizer.pad_token is None:            # match the original load-cell setup
    tokenizer.pad_token = tokenizer.eos_token
print("Reloaded model + processor from", SAVE_DIR)


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Reloaded model + processor from aigc_detector_qwen
